In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

# Load CSVs
metrics = pd.read_csv("metric.csv")
traces = pd.read_csv("trace.csv")

# Ensure timestamp is integer
metrics['timestamp'] = metrics['timestamp'].astype(int)
traces['timestamp'] = traces['timestamp'].astype(int)

# Define UTC+8 window: May 26, 2020 03:30 to 04:00 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
# convert to POSIX seconds (UTC) for comparison with epoch timestamps
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# Helper to build summary for a DataFrame given group keys and name of metric column
def build_summary(df, group_keys, value_col, name_col_label):
    # global p95 per group (calculate on full series before window filtering)
    grp = df.groupby(group_keys)[value_col]
    total_points = grp.count().rename('total_points')
    p95 = grp.quantile(0.95).rename('p95_threshold')
    # count points in the time window (>= start, < end)
    in_window = ((df['timestamp'] >= start_ts) & (df['timestamp'] < end_ts)).astype(int)
    points_in_window = df.assign(in_window=in_window).groupby(group_keys)['in_window'].sum().rename('points_in_window')
    # assemble
    summary = pd.concat([total_points, p95, points_in_window], axis=1).reset_index()
    # tidy types and sorting
    summary['total_points'] = summary['total_points'].astype(int)
    summary = summary.sort_values('total_points', ascending=False).reset_index(drop=True)
    # limit top 50 as requested
    summary = summary.head(50)
    return summary

# Build metric summary
metrics_summary = build_summary(metrics, ['cmdb_id', 'kpi_name'], 'value', 'kpi_name')

# Build trace summary
traces_summary = build_summary(traces, ['cmdb_id', 'trace_name'], 'value', 'trace_name')

# Keep output compact: select the requested columns only
metrics_summary = metrics_summary[['cmdb_id', 'kpi_name', 'total_points', 'p95_threshold', 'points_in_window']]
traces_summary = traces_summary[['cmdb_id', 'trace_name', 'total_points', 'p95_threshold', 'points_in_window']]

# Display the two compact summary tables
metrics_summary, traces_summary

```
Out[1]:
```
```python
# Compose a plain-English summary using the previously computed summaries
top_metric = metrics_summary.iloc[0]
top_trace = traces_summary.iloc[0]

summary = (
    f"Metrics summary (top results):\n"
    f"- Top metric group: {top_metric['cmdb_id']} / {top_metric['kpi_name']} with "
    f"{top_metric['total_points']} total points, global P95={top_metric['p95_threshold']:.6g}, "
    f"{top_metric['points_in_window']} points inside the 2020-05-26 03:30–04:00 (UTC+8) window.\n\n"
    f"Other high-volume metric groups also have ~60–61 total points and generally {metrics_summary['points_in_window'].mode().iloc[0]} points in the window (many groups have all window points present).\n\n"
    f"Traces summary (top results):\n"
    f"- Top trace group: {top_trace['cmdb_id']} / {top_trace['trace_name']} with "
    f"{top_trace['total_points']} total points, global P95={top_trace['p95_threshold']:.6g}, "
    f"{top_trace['points_in_window']} points inside the same window.\n\n"
    f"Many trace groups shown have 30 total points and 30 points in the window (i.e., the window covers the full available series for those traces). "
    f"P95 values vary by trace (examples above)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                  kpi_name  total_points  p95_threshold  points_in_window
0    os_012              Disk_io_util            61       1.000000                30
1    os_006                Send_total            61       0.000470                30
2    os_015  Incoming_network_traffic            61       0.003451                30
3    os_020              Disk_io_util            61       0.000000                30
4   osb_001              app.avg_time            61       0.010839                30
..      ...                       ...           ...            ...               ...
45   os_003             Disk_avgqu_sz            60       0.060964                30
46   os_003          Memory_available            60       8.640423                30
47   os_003  Incoming_network_traffic            60       2.422494                30
48   os_003               Disk_wr_ios            60      29.000000                30
49   os_002                Agent_ping            60       1.000000                30

[50 rows x 5 columns],        cmdb_id                                       trace_name  total_points  p95_threshold  points_in_window
0   docker_001    trace.from_os_022.remoteprocess.duration_mean            30       0.625199                30
1   docker_001     trace.from_os_022.remoteprocess.duration_p95            30       1.883025                30
2   docker_001       trace.from_os_022.remoteprocess.error_rate            30       0.000000                30
3   docker_001        trace.from_os_022.remoteprocess.row_count            30     110.550000                30
4   docker_001                     trace.self.csf.duration_mean            30       0.058816                30
..         ...                                              ...           ...            ...               ...
45  docker_002                    trace.self.local.duration_p95            30       0.756847                30
46  docker_002                      trace.self.local.error_rate            30       0.000000                30
47  docker_002                       trace.self.local.row_count            30    1622.650000                30
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean            30       0.030457                30
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95            30       0.046662                30

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing 'metrics' if present; otherwise load
try:
    metrics
except NameError:
    metrics = pd.read_csv("metric.csv")
    metrics['timestamp'] = metrics['timestamp'].astype(int)

# Ensure timestamp type
metrics['timestamp'] = metrics['timestamp'].astype(int)

# Define UTC+8 window explicitly (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Compute global group-level stats BEFORE filtering (total_points, p95_threshold)
grp = metrics.groupby(['cmdb_id', 'kpi_name'])['value']
group_total = grp.count().rename('total_points')
group_p95 = grp.quantile(0.95).rename('p95_threshold')
group_stats = pd.concat([group_total, group_p95], axis=1).reset_index()

# 2) Filter points in the specified window
window_df = metrics[(metrics['timestamp'] >= start_ts) & (metrics['timestamp'] < end_ts)].copy()

# 3) Aggregate window-level stats per group
points_in_window = window_df.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('points_in_window')
max_in_window = window_df.groupby(['cmdb_id', 'kpi_name'])['value'].max().rename('max_value_in_window')

# 4) Identify anomalies in the window: value > p95_threshold (use group-level p95)
# Merge window records with group p95
window_with_p95 = window_df.merge(group_p95.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies where value > p95_threshold
window_with_p95['is_anomaly'] = window_with_p95['value'] > window_with_p95['p95_threshold']

# Aggregate anomaly counts and earliest anomaly timestamp
anomaly_count = window_with_p95.groupby(['cmdb_id', 'kpi_name'])['is_anomaly'].sum().rename('anomaly_count')
earliest_anomaly_ts = (
    window_with_p95[window_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_ts')
)

# 5) Assemble final summary by merging all aggregated pieces
summary = group_stats.merge(points_in_window.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(max_in_window.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(anomaly_count.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(earliest_anomaly_ts.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for groups with no window points
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

# 6) Compute severity_ratio per rules: (max_value_in_window - p95_threshold)/p95_threshold if p95_threshold>0 else null
def compute_severity(row):
    p95 = row['p95_threshold']
    maxv = row['max_value_in_window']
    if pd.isna(maxv):
        return np.nan
    if pd.isna(p95) or p95 <= 0:
        return np.nan
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# 7) Convert earliest_anomaly_ts to UTC+8 localized human-readable string, keep nulls as NaN
def ts_to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return datetime.fromtimestamp(int(ts), pytz.UTC).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary['earliest_anomaly_ts'] = summary['earliest_anomaly_ts'].apply(ts_to_local_str)

# 8) Filter to rows with anomaly_count > 0
anomalies_summary = summary[summary['anomaly_count'] > 0].copy()

# 9) Sort and limit top 20 by anomaly_count desc then severity_ratio desc
anomalies_summary = anomalies_summary.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)

# 10) Keep output compact: select requested fields and concise types
anomalies_summary = anomalies_summary[
    ['cmdb_id', 'kpi_name', 'total_points', 'p95_threshold', 'points_in_window',
     'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio']
].reset_index(drop=True)

# Display final compact anomalies_summary (top 20)
anomalies_summary

```
Out[2]:
```
```python
summary = (
    "Anomaly detection summary (May 26, 2020 03:30–04:00 UTC+8):\n"
    "- Found 20 metric groups with anomaly_count > 0 within the window. Each reported group has anomaly_count = 3.\n"
    "- Top 3 by severity_ratio:\n"
    "  1) os_007 / Disk_wr_kbs — severity_ratio 29.95, p95=0.004438, max_in_window=0.137378, earliest_anomaly=2020-05-26 03:30:00 CST\n"
    "  2) db_011 / Call_Per_Sec — severity_ratio 6.77, p95=17.235, max_in_window=134.0, earliest_anomaly=2020-05-26 03:32:00 CST\n"
    "  3) os_017 / Disk_await — severity_ratio 3.73, p95=3.2425, max_in_window=15.33, earliest_anomaly=2020-05-26 03:39:00 CST\n"
    "- Other notable groups (disk I/O, CPU iowait, DB write/ops, Redis ops) also show >P95 spikes and should be reviewed.\n"
    "Recommendation: prioritize investigation on os_007 (Disk write KB/s), db_011 (Call_Per_Sec), and os_017 (Disk await) first, then examine other disk/IO and DB-related groups listed in the anomalies for root-cause analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name  total_points  p95_threshold  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  severity_ratio
0      os_007                Disk_wr_kbs            57       0.004438                29              3  2020-05-26 03:30:00 CST             0.137378       29.953540
1      db_011               Call_Per_Sec            60      17.235000                30              3  2020-05-26 03:32:00 CST           134.000000        6.774877
2      os_017                 Disk_await            60       3.242500                30              3  2020-05-26 03:39:00 CST            15.330000        3.727833
3      os_008                 Disk_svctm            60       1.515000                30              3  2020-05-26 03:36:00 CST             6.250000        3.125413
4      os_015              Disk_avgqu_sz            60       0.011537                30              3  2020-05-26 03:38:00 CST             0.046733        3.050531
5      os_015                Disk_wr_kbs            60       0.047616                30              3  2020-05-26 03:38:00 CST             0.178162        2.741610
6      os_016                Disk_wr_kbs            60       0.052154                30              3  2020-05-26 03:42:00 CST             0.179232        2.436582
7      os_019                 Disk_await            60       7.340500                30              3  2020-05-26 03:34:00 CST            25.110000        2.420748
8      os_004            CPU_iowait_time            60       0.838776                30              3  2020-05-26 03:30:00 CST             2.827468        2.370946
9      os_011              Disk_avgqu_sz            58       0.016396                29              3  2020-05-26 03:35:00 CST             0.052700        2.214287
10     db_003        LFParaWrite_Per_Sec            60       1.462500                30              3  2020-05-26 03:44:00 CST             3.750000        1.564103
11  redis_008  instantaneous_ops_per_sec            60       3.150000                30              3  2020-05-26 03:34:00 CST             8.000000        1.539683
12     os_020            CPU_iowait_time            60       0.010507                30              3  2020-05-26 03:30:00 CST             0.026439        1.516215
13     db_010        LFParaWrite_Per_Sec            60       0.152000                30              3  2020-05-26 03:40:00 CST             0.360000        1.368421
14     os_019                Disk_wr_ios            60       3.100000                30              3  2020-05-26 03:42:00 CST             7.000000        1.258065
15     os_022                 Disk_await            60       0.156500                30              3  2020-05-26 03:39:00 CST             0.330000        1.108626
16     os_018                 Disk_svctm            60       6.148500                30              3  2020-05-26 03:30:00 CST            12.250000        0.992356
17     os_016                Disk_wr_ios            60       6.050000                30              3  2020-05-26 03:42:00 CST            11.000000        0.818182
18     os_018                    Page_po            60     184.950000                30              3  2020-05-26 03:35:00 CST           331.000000        0.789673
19     os_018                Disk_wr_kbs            60       0.175339                30              3  2020-05-26 03:36:00 CST             0.291769        0.664024```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing metrics, group_p95, start_ts, end_ts if available; otherwise load/compute
try:
    metrics
except NameError:
    metrics = pd.read_csv("metric.csv")
    metrics['timestamp'] = metrics['timestamp'].astype(int)

metrics['timestamp'] = metrics['timestamp'].astype(int)

tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 per (cmdb_id, kpi_name) computed on full series (rule #9)
group_p95 = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_threshold').reset_index()
group_total = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('total_points').reset_index()

# 2) Filter points to the incident window (inclusive start, exclusive end)
# Reuse start_ts and end_ts if present; otherwise compute
try:
    start_ts, end_ts
except NameError:
    start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
    end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
    start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
    end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

window_df = metrics[(metrics['timestamp'] >= start_ts) & (metrics['timestamp'] < end_ts)].copy()

# 3) Attach group-level p95 to window points
window_df = window_df.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Flag anomaly points in window where value > p95_threshold
window_df['is_anomaly'] = window_df['value'] > window_df['p95_threshold']

# 5) For each (cmdb_id,kpi_name) with any anomaly in window, identify consecutive runs of anomaly points
# We'll work only on anomaly points to find runs
anom_points = window_df[window_df['is_anomaly']].copy()
if anom_points.empty:
    runs_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95_threshold','run_start_ts_utc+8','run_end_ts_utc+8',
        'run_length_minutes','max_value_in_run','severity_ratio_max','anomaly_count_in_window','total_points'
    ])
else:
    # sort to ensure chronological order
    anom_points = anom_points.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).reset_index(drop=True)

    # compute difference between consecutive anomaly timestamps per group
    anom_points['prev_ts'] = anom_points.groupby(['cmdb_id', 'kpi_name'])['timestamp'].shift(1)
    anom_points['ts_diff'] = anom_points['timestamp'] - anom_points['prev_ts']

    # mark start of a new run when ts_diff != 60 (non-consecutive) or prev is NaN
    anom_points['new_run'] = ((anom_points['ts_diff'] != 60) | anom_points['ts_diff'].isna()).astype(int)

    # assign run ids within each group by cumulative sum of new_run
    anom_points['run_id'] = anom_points.groupby(['cmdb_id', 'kpi_name'])['new_run'].cumsum()

    # aggregate per run
    runs = anom_points.groupby(['cmdb_id', 'kpi_name', 'p95_threshold', 'run_id']).agg(
        run_start_ts=('timestamp', 'min'),
        run_end_ts=('timestamp', 'max'),
        run_length_minutes=('timestamp', 'count'),
        max_value_in_run=('value', 'max')
    ).reset_index()

    # keep only earliest run per (cmdb_id,kpi_name) -> i.e., smallest run_start_ts
    earliest_runs = runs.sort_values(['cmdb_id', 'kpi_name', 'run_start_ts']).groupby(
        ['cmdb_id', 'kpi_name'], as_index=False
    ).first()

    # compute severity_ratio_max per rules (null if p95_threshold <= 0)
    def compute_sev(row):
        p95 = row['p95_threshold']
        maxv = row['max_value_in_run']
        if pd.isna(maxv) or pd.isna(p95) or p95 <= 0:
            return np.nan
        return (maxv - p95) / p95

    earliest_runs['severity_ratio_max'] = earliest_runs.apply(compute_sev, axis=1)

    # anomaly_count_in_window per group (count of anomaly points in the whole window)
    anomaly_count = anom_points.groupby(['cmdb_id', 'kpi_name'])['is_anomaly'].sum().rename('anomaly_count_in_window').reset_index()

    # total_points from full series
    # reuse group_total computed earlier
    # assemble final summary
    runs_summary = earliest_runs.merge(anomaly_count, on=['cmdb_id', 'kpi_name'], how='left')
    runs_summary = runs_summary.merge(group_total, on=['cmdb_id', 'kpi_name'], how='left')

    # convert run_start/end timestamps to UTC+8 readable strings
    def ts_to_local_str(ts):
        return datetime.fromtimestamp(int(ts), pytz.UTC).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    runs_summary['run_start_ts_utc+8'] = runs_summary['run_start_ts'].apply(ts_to_local_str)
    runs_summary['run_end_ts_utc+8'] = runs_summary['run_end_ts'].apply(ts_to_local_str)

    # select and order columns as requested
    runs_summary = runs_summary[[
        'cmdb_id','kpi_name','p95_threshold','run_start_ts_utc+8','run_end_ts_utc+8',
        'run_length_minutes','max_value_in_run','severity_ratio_max','anomaly_count_in_window','total_points'
    ]]

    # Only include rows where anomaly_count_in_window > 0 (should already be true)
    runs_summary = runs_summary[runs_summary['anomaly_count_in_window'] > 0].copy()

    # Sort by severity_ratio_max desc (NaNs last) and limit to top 20
    runs_summary = runs_summary.sort_values('severity_ratio_max', ascending=False).head(20).reset_index(drop=True)

# Display compact result (top 20 runs)
runs_summary

```
Out[3]:
```
```python
# Compose a plain-English summary from the previously computed `runs_summary` DataFrame
try:
    runs_summary
except NameError:
    raise RuntimeError("runs_summary not found in the environment. Please run the anomaly detection cell first.")

# Keep summary compact: overall count and top 5 by severity_ratio_max
total_runs = len(runs_summary)
top5 = runs_summary.head(5)

lines = [
    f"Detected {total_runs} components/KPIs with >=1 anomalous run (value > global P95) during 2020-05-26 03:30–04:00 (UTC+8).",
    "Top 5 anomalous runs by severity_ratio_max:"
]
for i, row in top5.iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']} / {row['kpi_name']}: "
        f"P95={row['p95_threshold']}, run {row['run_start_ts_utc+8']} to {row['run_end_ts_utc+8']} "
        f"({int(row['run_length_minutes'])} min), max_in_run={row['max_value_in_run']}, "
        f"severity_ratio_max={row['severity_ratio_max']:.3f}, anomalies_in_window={int(row['anomaly_count_in_window'])}, "
        f"total_points={int(row['total_points'])}."
    )

lines.append("")
lines.append("Observations and recommended action:")
lines.append("- The highest severities are: db_002 Physical_Read_Per_Sec, os_007 Page_po, and os_006 Page_po (see top list).")
lines.append("- Many runs are short (1–2 minutes); a few span 2 minutes. Prioritize investigation of the highest severity runs first.")
lines.append("- Focus initial root-cause checks on disk I/O and page-in/page-out behavior for the OS nodes and on DB read/write hotspots for the DB entries listed.")
summary_text = "\n".join(lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name  p95_threshold       run_start_ts_utc+8         run_end_ts_utc+8  run_length_minutes  max_value_in_run  severity_ratio_max  anomaly_count_in_window  total_points
0       db_002      Physical_Read_Per_Sec       0.300000  2020-05-26 03:58:00 CST  2020-05-26 03:58:00 CST                   1         16.600000           54.333333                        1            60
1       os_007                    Page_po       4.050000  2020-05-26 03:42:00 CST  2020-05-26 03:42:00 CST                   1        143.000000           34.308642                        1            60
2       os_006                    Page_po       4.000000  2020-05-26 03:45:00 CST  2020-05-26 03:46:00 CST                   2        128.000000           31.000000                        2            60
3       os_006              Disk_avgqu_sz       0.001179  2020-05-26 03:45:00 CST  2020-05-26 03:45:00 CST                   1          0.031020           25.300394                        1            60
4       os_021                 Disk_await       0.178000  2020-05-26 03:48:00 CST  2020-05-26 03:48:00 CST                   1          4.000000           21.471910                        2            60
5       db_009               Redo_Per_Sec    2703.090000  2020-05-26 03:58:00 CST  2020-05-26 03:59:00 CST                   2      58278.000000           20.559770                        2            60
6       os_020                Disk_rd_kbs       0.000196  2020-05-26 03:57:00 CST  2020-05-26 03:57:00 CST                   1          0.003729           17.996434                        1            60
7       os_020                    Page_pi       0.204197  2020-05-26 03:57:00 CST  2020-05-26 03:57:00 CST                   1          3.853951           17.873709                        1            57
8       os_007              Disk_avgqu_sz       0.001350  2020-05-26 03:42:00 CST  2020-05-26 03:43:00 CST                   2          0.021657           15.042222                        2            60
9    redis_012  instantaneous_ops_per_sec       1.000000  2020-05-26 03:36:00 CST  2020-05-26 03:36:00 CST                   1         10.000000            9.000000                        1            60
10      db_001              Login_Per_Sec       0.200000  2020-05-26 03:49:00 CST  2020-05-26 03:50:00 CST                   2          1.600000            7.000000                        2            60
11      db_011               Call_Per_Sec      17.235000  2020-05-26 03:32:00 CST  2020-05-26 03:33:00 CST                   2        134.000000            6.774877                        3            60
12  docker_008         container_cpu_used       1.300000  2020-05-26 03:46:00 CST  2020-05-26 03:46:00 CST                   1          9.000000            5.923077                        2            58
13      os_007            CPU_iowait_time       0.005557  2020-05-26 03:43:00 CST  2020-05-26 03:43:00 CST                   1          0.027787            4.000360                        1            51
14      os_004              Disk_avgqu_sz       0.102181  2020-05-26 03:30:00 CST  2020-05-26 03:30:00 CST                   1          0.471112            3.610559                        2            60
15      os_018   Outgoing_network_traffic       0.033493  2020-05-26 03:45:00 CST  2020-05-26 03:45:00 CST                   1          0.143245            3.276911                        2            60
16     osb_001               app.avg_time       0.010839  2020-05-26 03:40:00 CST  2020-05-26 03:41:00 CST                   2          0.045127            3.163201                        2            61
17      os_015              Disk_avgqu_sz       0.011537  2020-05-26 03:38:00 CST  2020-05-26 03:38:00 CST                   1          0.046733            3.050531                        3            60
18      db_002              Login_Per_Sec       0.600000  2020-05-26 03:54:00 CST  2020-05-26 03:54:00 CST                   1          2.400000            3.000000                        1            60
19      os_010              Disk_avgqu_sz       0.000167  2020-05-26 03:33:00 CST  2020-05-26 03:33:00 CST                   1          0.000666            2.983254                        2            57```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing 'traces' if available, else load
try:
    traces
except NameError:
    traces = pd.read_csv("trace.csv")
    traces['timestamp'] = traces['timestamp'].astype(int)

traces['timestamp'] = traces['timestamp'].astype(int)

# Timezone and window (Asia/Shanghai == UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Global stats BEFORE filtering: total_points and p95_threshold per (cmdb_id, trace_name)
group_total = traces.groupby(['cmdb_id', 'trace_name'])['value'].count().rename('total_points').reset_index()
group_p95 = traces.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('p95_threshold').reset_index()

groups = group_total.merge(group_p95, on=['cmdb_id', 'trace_name'], how='outer')

# 2) Filter to incident window
window_traces = traces[(traces['timestamp'] >= start_ts) & (traces['timestamp'] < end_ts)].copy()

# 3) Compute points_in_window and max_value_in_window per group
points_in_window = window_traces.groupby(['cmdb_id', 'trace_name'])['value'].count().rename('points_in_window').reset_index()
max_in_window = window_traces.groupby(['cmdb_id', 'trace_name'])['value'].max().rename('max_value_in_window').reset_index()

# 4) Determine anomaly points (value > p95_threshold) using global p95
# Merge p95 into the windowed points
window_with_p95 = window_traces.merge(group_p95, on=['cmdb_id', 'trace_name'], how='left')
window_with_p95['is_anomaly'] = window_with_p95['value'] > window_with_p95['p95_threshold']

# anomaly_count and earliest anomaly timestamp per group
anomaly_count = window_with_p95.groupby(['cmdb_id', 'trace_name'])['is_anomaly'].sum().rename('anomaly_count').reset_index()
earliest_anom_ts = (
    window_with_p95[window_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_ts')
    .reset_index()
)

# 5) Assemble summary by merging aggregates
summary = groups.merge(points_in_window, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(max_in_window, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(anomaly_count, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(earliest_anom_ts, on=['cmdb_id', 'trace_name'], how='left')

# Fill NaNs for counts
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

# 6) Compute severity_ratio per rules
def compute_severity(p95, maxv):
    if pd.isna(maxv) or pd.isna(p95) or p95 <= 0:
        return np.nan
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(lambda r: compute_severity(r['p95_threshold'], r['max_value_in_window']), axis=1)

# 7) Convert earliest_anomaly_ts to readable UTC+8 string, keep nulls as <NA>
def ts_to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return datetime.fromtimestamp(int(ts), pytz.UTC).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary['earliest_anomaly_ts'] = summary['earliest_anomaly_ts'].apply(ts_to_local_str)

# 8) Filter to rows with anomaly_count > 0, sort and limit top 20 by anomaly_count desc then severity_ratio desc
traces_anomalies_summary = summary[summary['anomaly_count'] > 0].copy()
traces_anomalies_summary = traces_anomalies_summary.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)

# 9) Select and order the requested output columns
traces_anomalies_summary = traces_anomalies_summary[[
    'cmdb_id', 'trace_name', 'total_points', 'p95_threshold', 'points_in_window',
    'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio'
]].reset_index(drop=True)

# Display compact table (top 20)
traces_anomalies_summary

```
Out[4]:
```
```python
# Compose a plain-English summary from the previously computed `traces_anomalies_summary` DataFrame
try:
    traces_anomalies_summary
except NameError:
    raise RuntimeError("traces_anomalies_summary not found. Please run the trace anomaly detection cell first.")

total_rows = len(traces_anomalies_summary)
top5 = traces_anomalies_summary.head(5)

lines = [
    f"Trace anomaly summary (May 26, 2020 03:30–04:00 UTC+8):",
    f"- Found {total_rows} trace groups with anomalies (value > global P95) in the window.",
    "- Top 5 by ordering (anomaly_count desc, then severity_ratio):"
]

for i, row in top5.iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']} / {row['trace_name']}: "
        f"total_points={int(row['total_points'])}, p95={row['p95_threshold']}, "
        f"points_in_window={int(row['points_in_window'])}, anomalies={int(row['anomaly_count'])}, "
        f"earliest={row['earliest_anomaly_ts']}, max_in_window={row['max_value_in_window']}, "
        f"severity_ratio={row['severity_ratio']:.3f}"
    )

lines.append("")
lines.append("Observation: Several docker/os groups show clear spikes above their global P95 (notably docker_002 and os_022 entries).")
lines.append("Recommendation: Investigate the listed docker_002 and os_022 trace groups (high duration and error_rate spikes) first, then review other top entries for related service or network issues.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  total_points  p95_threshold  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  severity_ratio
0   docker_002                      trace.self.csf.duration_p95            30       1.905797                30              2  2020-05-26 03:40:00 CST            60.548250       30.770558
1       os_022                   trace.from_root.osb.error_rate            30       0.005238                30              2  2020-05-26 03:40:00 CST             0.133333       24.453973
2       os_022                        trace.self.csf.error_rate            30       0.005238                30              2  2020-05-26 03:40:00 CST             0.133333       24.453973
3   docker_002                        trace.self.csf.error_rate            30       0.009116                30              2  2020-05-26 03:40:00 CST             0.200000       20.938845
4   docker_002                     trace.self.csf.duration_mean            30       1.345209                30              2  2020-05-26 03:40:00 CST            18.069200       12.432265
5   docker_001                     trace.self.jdbc.duration_p95            30       0.007978                30              2  2020-05-26 03:35:00 CST             0.032700        3.099029
6   docker_002    trace.from_os_022.remoteprocess.duration_mean            30      14.373460                30              2  2020-05-26 03:40:00 CST            44.472083        2.094042
7       os_022  trace.to_docker_002.remoteprocess.duration_mean            30      14.373460                30              2  2020-05-26 03:40:00 CST            44.472083        2.094042
8       os_022                     trace.self.csf.duration_mean            30       7.599660                30              2  2020-05-26 03:40:00 CST            22.083400        1.905841
9       os_022                trace.from_root.osb.duration_mean            30       7.622447                30              2  2020-05-26 03:40:00 CST            22.128933        1.903127
10      os_022                      trace.self.csf.duration_p95            30      44.924675                30              2  2020-05-26 03:40:00 CST            86.250800        0.919898
11      os_022                 trace.from_root.osb.duration_p95            30      44.942600                30              2  2020-05-26 03:40:00 CST            86.266600        0.919484
12  docker_002     trace.from_os_022.remoteprocess.duration_p95            30      47.904187                30              2  2020-05-26 03:40:00 CST            87.099850        0.818210
13      os_022   trace.to_docker_002.remoteprocess.duration_p95            30      47.904187                30              2  2020-05-26 03:40:00 CST            87.099850        0.818210
14  docker_002               trace.self.flyremote.duration_mean            30       0.125865                30              2  2020-05-26 03:42:00 CST             0.205187        0.630216
15  docker_008                    trace.self.jdbc.duration_mean            30       0.002574                30              2  2020-05-26 03:43:00 CST             0.004189        0.627681
16      os_021                trace.from_root.osb.duration_mean            30       0.666058                30              2  2020-05-26 03:41:00 CST             1.070806        0.607676
17  docker_001                   trace.self.local.duration_mean            30       0.014530                30              2  2020-05-26 03:46:00 CST             0.023321        0.605035
18  docker_007                    trace.self.jdbc.duration_mean            30       0.002640                30              2  2020-05-26 03:43:00 CST             0.004221        0.598743
19  docker_001                trace.self.flyremote.duration_p95            30       0.056275                30              2  2020-05-26 03:38:00 CST             0.089450        0.589516```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing variables if present, else load/compute
try:
    metrics
except NameError:
    metrics = pd.read_csv("metric.csv")
    metrics['timestamp'] = metrics['timestamp'].astype(int)

metrics['timestamp'] = metrics['timestamp'].astype(int)

tz = pytz.timezone('Asia/Shanghai')

# Reuse start_ts/end_ts if available; otherwise compute (UTC+8 window)
try:
    start_ts, end_ts
except NameError:
    start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
    end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
    start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
    end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Global P95 and total_points per (cmdb_id, kpi_name) computed on full series (before filtering)
group_p95 = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_threshold').reset_index()
group_total = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('total_points').reset_index()

# 2) Filter points to the incident window
window_df = metrics[(metrics['timestamp'] >= start_ts) & (metrics['timestamp'] < end_ts)].copy()

# 3) For os_022 only: get points_in_window and max_value_in_window
window_os022 = window_df[window_df['cmdb_id'] == 'os_022'].copy()

points_in_window = window_os022.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('points_in_window').reset_index()
max_in_window = window_os022.groupby(['cmdb_id', 'kpi_name'])['value'].max().rename('max_value_in_window').reset_index()

# 4) Merge global p95 and totals into a per-kpi summary for os_022
summary = group_total.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='inner')
summary = summary[summary['cmdb_id'] == 'os_022'].merge(points_in_window, on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(max_in_window, on=['cmdb_id', 'kpi_name'], how='left')

summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)

# 5) Determine anomaly_count and earliest_anomaly_ts using window points and global p95
# Merge p95 into window records for os_022 and flag anomalies
window_os022 = window_os022.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='left')
window_os022['is_anomaly'] = window_os022['value'] > window_os022['p95_threshold']

anomaly_count = window_os022.groupby(['cmdb_id', 'kpi_name'])['is_anomaly'].sum().rename('anomaly_count').reset_index()
earliest_anom = (
    window_os022[window_os022['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_ts')
    .reset_index()
)

# 6) Merge anomaly aggregates into summary
summary = summary.merge(anomaly_count, on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

# 7) Compute severity_ratio per rules
def compute_severity(row):
    p95 = row['p95_threshold']
    maxv = row['max_value_in_window']
    if pd.isna(maxv) or pd.isna(p95) or p95 <= 0:
        return np.nan
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# 8) Convert earliest_anomaly_ts to readable UTC+8 string, keep NA when missing
def ts_to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return datetime.fromtimestamp(int(ts), pytz.UTC).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary['earliest_anomaly_ts'] = summary['earliest_anomaly_ts'].apply(ts_to_local_str)

# 9) Filter to anomaly_count > 0, sort by severity_ratio desc, limit top 20, and select requested columns
os022_anomalies = summary[summary['anomaly_count'] > 0].copy()
os022_anomalies = os022_anomalies.sort_values('severity_ratio', ascending=False).head(20)

os022_anomalies = os022_anomalies[[
    'cmdb_id', 'kpi_name', 'total_points', 'p95_threshold', 'points_in_window',
    'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio'
]].reset_index(drop=True)

# Display the compact table for os_022 anomalies
os022_anomalies

```
Out[5]:
```
```python
# Compose a concise plain-English summary for os_022 anomalies using the previously computed os022_anomalies
try:
    os022_anomalies
except NameError:
    raise RuntimeError("os022_anomalies not found. Please run the prior cell to compute os_022 anomalies.")

count = len(os022_anomalies)
top3 = os022_anomalies.head(3)

lines = [
    f"Summary for cmdb_id = 'os_022' during 2020-05-26 03:30–04:00 (UTC+8):",
    f"- Found {count} KPI(s) with values exceeding their global P95 in the window.",
    "- Top 3 by severity_ratio:"
]

for i, row in top3.iterrows():
    lines.append(
        f"  {i+1}) {row['kpi_name']}: anomaly_count={int(row['anomaly_count'])}, "
        f"P95={row['p95_threshold']}, max_in_window={row['max_value_in_window']}, "
        f"earliest_anomaly={row['earliest_anomaly_ts']}, severity_ratio={row['severity_ratio']:.3f}"
    )

lines += [
    "",
    "Notable findings:",
    "- Highest severity is Disk_await (severity ~1.11, 3 anomalies) — indicates elevated disk latency spikes around 03:39.",
    "- Network I/O (Send_total, Recv_total) show spikes as well; check network throughput and packet activity.",
    "- ss_total and Memory_used_pct also had 3 anomaly points — review session counts and memory usage patterns.",
    "",
    "Recommendation: Prioritize investigation of disk I/O/latency (Disk_await, Disk_wr_ios, Disk_wr_kbs) and network spikes (Send/Recv) on os_022, then review session/memory metrics (ss_total, Memory_used_pct) for correlated impact."
]

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name  total_points  p95_threshold  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  severity_ratio
0   os_022                Disk_await            60       0.156500                30              3  2020-05-26 03:39:00 CST             0.330000        1.108626
1   os_022                Send_total            60       0.057386                30              2  2020-05-26 03:45:00 CST             0.082403        0.435941
2   os_022                Recv_total            60      36.709276                30              1  2020-05-26 03:45:00 CST            52.008074        0.416756
3   os_022               Disk_wr_ios            60       6.000000                30              1  2020-05-26 03:30:00 CST             7.000000        0.166667
4   os_022                Disk_svctm            60       3.349000                30              2  2020-05-26 03:45:00 CST             3.870000        0.155569
5   os_022             CPU_user_time            60       4.109143                30              1  2020-05-26 03:56:00 CST             4.462798        0.086065
6   os_022               Disk_wr_kbs            60       1.321358                30              2  2020-05-26 03:34:00 CST             1.375531        0.040998
7   os_022  Incoming_network_traffic            60       0.036085                30              1  2020-05-26 03:59:00 CST             0.037502        0.039261
8   os_022  Outgoing_network_traffic            60       0.056128                30              2  2020-05-26 03:52:00 CST             0.057110        0.017489
9   os_022           CPU_system_time            60       0.305729                30              1  2020-05-26 03:56:00 CST             0.310822        0.016658
10  os_022          Received_packets            12     143.350000                 6              1  2020-05-26 03:59:00 CST           145.000000        0.011510
11  os_022              Sent_packets            12     144.350000                 6              1  2020-05-26 03:59:00 CST           146.000000        0.011431
12  os_022                  ss_total            60     362.050000                30              3  2020-05-26 03:39:00 CST           365.000000        0.008148
13  os_022                   Page_po            60    1377.150000                30              1  2020-05-26 03:33:00 CST          1380.000000        0.002069
14  os_022               Memory_free            12       0.300331                 6              1  2020-05-26 03:59:00 CST             0.300541        0.000700
15  os_022              FS_max_avail            12       8.634627                 6              1  2020-05-26 03:45:00 CST             8.637850        0.000373
16  os_022              CPU_idle_pct            60      96.333255                30              1  2020-05-26 03:42:00 CST            96.360530        0.000283
17  os_022           Memory_used_pct            60      56.430500                30              3  2020-05-26 03:30:00 CST            56.440000        0.000168
18  os_022               Memory_used            60      35.511260                30              2  2020-05-26 03:30:00 CST            35.514565        0.000093```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing 'metrics' if present, otherwise load
try:
    metrics
except NameError:
    metrics = pd.read_csv("metric.csv")
    metrics['timestamp'] = metrics['timestamp'].astype(int)

metrics['timestamp'] = metrics['timestamp'].astype(int)

# Timezone and window (Asia/Shanghai == UTC+8)
tz = pytz.timezone('Asia/Shanghai')
try:
    start_ts, end_ts
except NameError:
    start_local = tz.localize(datetime(2020, 5, 26, 3, 30, 0))
    end_local = tz.localize(datetime(2020, 5, 26, 4, 0, 0))
    start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
    end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 1) Global P95 and total_points per (cmdb_id, kpi_name) computed on full series BEFORE filtering
group_p95 = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_threshold').reset_index()
group_total = metrics.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('total_points').reset_index()

# 2) Filter points to the incident window and for cmdb_id == 'docker_002' only
window_df = metrics[(metrics['timestamp'] >= start_ts) & (metrics['timestamp'] < end_ts)].copy()
window_docker002 = window_df[window_df['cmdb_id'] == 'docker_002'].copy()

# 3) Compute points_in_window and max_value_in_window per kpi for docker_002
points_in_window = window_docker002.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('points_in_window').reset_index()
max_in_window = window_docker002.groupby(['cmdb_id', 'kpi_name'])['value'].max().rename('max_value_in_window').reset_index()

# 4) Assemble base summary for docker_002 by merging global stats with window aggregates
summary = group_total.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='inner')
summary = summary[summary['cmdb_id'] == 'docker_002'].merge(points_in_window, on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(max_in_window, on=['cmdb_id', 'kpi_name'], how='left')
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)

# 5) Determine anomaly_count and earliest_anomaly_ts by comparing window points to global p95
window_docker002 = window_docker002.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='left')
window_docker002['is_anomaly'] = window_docker002['value'] > window_docker002['p95_threshold']

anomaly_count = window_docker002.groupby(['cmdb_id', 'kpi_name'])['is_anomaly'].sum().rename('anomaly_count').reset_index()
earliest_anom = (
    window_docker002[window_docker002['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_ts')
    .reset_index()
)

summary = summary.merge(anomaly_count, on=['cmdb_id', 'kpi_name'], how='left')
summary = summary.merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

# 6) Compute severity_ratio per rules
def compute_severity(p95, maxv):
    if pd.isna(maxv) or pd.isna(p95) or p95 <= 0:
        return np.nan
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(lambda r: compute_severity(r['p95_threshold'], r['max_value_in_window']), axis=1)

# 7) Convert earliest_anomaly_ts to readable UTC+8 string, keep NA when missing
def ts_to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return datetime.fromtimestamp(int(ts), pytz.UTC).astimezone(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

summary['earliest_anomaly_ts'] = summary['earliest_anomaly_ts'].apply(ts_to_local_str)

# 8) Filter to anomaly_count > 0, sort by severity_ratio desc, limit top 20, and select requested columns
docker002_anomalies = summary[summary['anomaly_count'] > 0].copy()
docker002_anomalies = docker002_anomalies.sort_values('severity_ratio', ascending=False).head(20)

docker002_anomalies = docker002_anomalies[[
    'cmdb_id', 'kpi_name', 'total_points', 'p95_threshold', 'points_in_window',
    'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'severity_ratio'
]].reset_index(drop=True)

# Display compact aggregated table for docker_002 anomalies
docker002_anomalies

```
Out[6]:
```
```python
# Summarize docker_002 anomalies succinctly
try:
    df = docker002_anomalies
except NameError:
    raise RuntimeError("docker002_anomalies not found in the environment. Please run the prior analysis cell first.")

if df.empty:
    summary_text = "No anomalies detected for cmdb_id='docker_002' in the window 2020-05-26 03:30–04:00 (UTC+8)."
else:
    row = df.iloc[0]
    summary_text = (
        "Summary for cmdb_id='docker_002' during 2020-05-26 03:30–04:00 (UTC+8):\n"
        "- Only one KPI exceeded its global P95 in the window:\n"
        f"  * container_cpu_used: total_points={int(row['total_points'])}, global P95={row['p95_threshold']}, "
        f"points_in_window={int(row['points_in_window'])}, anomaly_count={int(row['anomaly_count'])},\n"
        f"    earliest_anomaly={row['earliest_anomaly_ts']}, max_value_in_window={row['max_value_in_window']}, "
        f"severity_ratio={row['severity_ratio']:.3f}\n\n"
        "Recommendation: Investigate CPU usage for container 'docker_002' (container_cpu_used) around 03:47 (UTC+8) — "
        "this single spike (max 57.0 vs P95 35.25) suggests a notable CPU burst that may impact service performance."
    )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id            kpi_name  total_points  p95_threshold  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  severity_ratio
0  docker_002  container_cpu_used            56          35.25                28              1  2020-05-26 03:47:00 CST                 57.0        0.617021```
```